# Real-Data Orbital Congestion Monitoring

This is the **single master notebook** for Coco/Ririko's GitHub, Streamlit dashboard, and research paper.

## Important revision

This version replaces the earlier 28-record demonstration workflow. It uses a **real CelesTrak SATCAT catalogue snapshot** and can refresh the catalogue directly from CelesTrak. Any paper values produced by the old demonstration dataset, including the earlier 88% accuracy, 0.86 silhouette score, 96.77% PCA variance, and 28-record sample size, must be replaced with the outputs generated by this notebook.

## Scientific scope

The workflow identifies objects that occupy similar orbital regimes and ranks local crowding in standardized orbital feature space. It does **not** calculate collision probability, propagate trajectories through time, or perform conjunction analysis.


## 1. Research question and completed project workflow

**Research question:** Can real orbital catalogue features be used to identify groups of satellites and debris that occupy similar Low Earth Orbit regimes and provide a transparent, simplified measure of local feature-space congestion?

This notebook completes the technical work required by the project timeline:

1. real CelesTrak data acquisition and validation,
2. data cleaning and reproducible filtering,
3. orbital feature engineering,
4. feature scaling,
5. pilot-versus-full-catalogue comparison,
6. K-Means clustering and silhouette evaluation,
7. PCA visualization,
8. nearest-neighbour analysis,
9. a percentile-based congestion score,
10. debris-to-payload proximity screening,
11. owner-level descriptive summaries,
12. exploratory Random Forest classification,
13. paper-ready figures and tables, and
14. Streamlit-compatible exports.


## 2. Imports and reproducibility settings


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import json
import os
import warnings

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import requests

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11


## 3. Repository paths and analysis configuration


In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
OUTPUT_FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'

for folder in (DATA_DIR, OUTPUT_TABLE_DIR, OUTPUT_FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SATCAT_URL = 'https://celestrak.org/pub/satcat.csv'
LOCAL_SNAPSHOT_PATH = DATA_DIR / 'satcat.csv'
REFRESH_FROM_WEB = True
REQUEST_TIMEOUT_SECONDS = 60

LEO_MIN_ALTITUDE_KM = 160
LEO_MAX_ALTITUDE_KM = 2000
FEATURE_COLUMNS = [
    'PERIOD',
    'INCLINATION',
    'MEAN_ALTITUDE',
    'ORBIT_SPREAD',
    'ALTITUDE_RATIO',
]
K_VALUES = list(range(2, 9))
REQUESTED_NEIGHBORS = 5
PILOT_SAMPLE_SIZE = 30
SILHOUETTE_SAMPLE_SIZE = 10_000

print('Project root:', PROJECT_ROOT)
print('Real SATCAT snapshot:', LOCAL_SNAPSHOT_PATH)
print('Table outputs:', OUTPUT_TABLE_DIR)
print('Figure outputs:', OUTPUT_FIGURE_DIR)


Project root: /mnt/data/coco_orbital_congestion_real_data
Real SATCAT snapshot: /mnt/data/coco_orbital_congestion_real_data/data/satcat.csv
Table outputs: /mnt/data/coco_orbital_congestion_real_data/outputs/tables
Figure outputs: /mnt/data/coco_orbital_congestion_real_data/outputs/figures


## 4. Load real CelesTrak SATCAT data

The official source is CelesTrak's current SATCAT CSV:

- Catalogue page: https://celestrak.org/satcat/
- CSV documentation: https://celestrak.org/satcat/satcat-format.php
- Raw CSV: https://celestrak.org/pub/satcat.csv

The notebook attempts one live refresh. If the request fails, it uses the committed `data/satcat.csv` snapshot. The fallback is still real CelesTrak data. This notebook never substitutes synthetic or embedded teaching data.


In [3]:
EXPECTED_COLUMNS = [
    'OBJECT_NAME', 'OBJECT_ID', 'NORAD_CAT_ID', 'OBJECT_TYPE',
    'OPS_STATUS_CODE', 'OWNER', 'LAUNCH_DATE', 'LAUNCH_SITE',
    'DECAY_DATE', 'PERIOD', 'INCLINATION', 'APOGEE', 'PERIGEE',
    'RCS', 'DATA_STATUS_CODE', 'ORBIT_CENTER', 'ORBIT_TYPE',
]


def file_sha256(path: Path) -> str:
    digest = sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def load_real_satcat() -> tuple[pd.DataFrame, dict]:
    source_mode = None
    error_message = None

    if REFRESH_FROM_WEB:
        try:
            response = requests.get(
                SATCAT_URL,
                timeout=REQUEST_TIMEOUT_SECONDS,
                headers={'User-Agent': 'Coco-Orbital-Congestion-Research/1.0'},
            )
            response.raise_for_status()
            if len(response.content) < 100_000:
                raise ValueError('Downloaded file was unexpectedly small.')
            LOCAL_SNAPSHOT_PATH.write_bytes(response.content)
            source_mode = 'live download from CelesTrak'
        except Exception as exc:
            error_message = f'{type(exc).__name__}: {exc}'

    if not LOCAL_SNAPSHOT_PATH.exists():
        raise FileNotFoundError(
            'Real SATCAT data could not be downloaded and data/satcat.csv is missing. '
            'Download https://celestrak.org/pub/satcat.csv and save it as data/satcat.csv.'
        )

    if source_mode is None:
        source_mode = 'committed real CelesTrak snapshot'

    frame = pd.read_csv(LOCAL_SNAPSHOT_PATH, low_memory=False)
    frame.columns = [str(column).strip() for column in frame.columns]

    provenance = {
        'source_url': SATCAT_URL,
        'source_mode': source_mode,
        'loaded_utc': datetime.now(timezone.utc).isoformat(),
        'local_file': str(LOCAL_SNAPSHOT_PATH.relative_to(PROJECT_ROOT)),
        'sha256': file_sha256(LOCAL_SNAPSHOT_PATH),
        'live_refresh_error': error_message,
        'raw_rows': int(len(frame)),
        'raw_columns': int(frame.shape[1]),
    }
    return frame, provenance


df_raw, provenance = load_real_satcat()

missing_columns = [column for column in EXPECTED_COLUMNS if column not in df_raw.columns]
if missing_columns:
    raise ValueError(f'SATCAT file is missing required columns: {missing_columns}')

print(json.dumps(provenance, indent=2))
print('Raw shape:', df_raw.shape)
df_raw.head()


{
  "source_url": "https://celestrak.org/pub/satcat.csv",
  "source_mode": "committed real CelesTrak snapshot",
  "loaded_utc": "2026-07-22T04:00:22.902811+00:00",
  "local_file": "data/satcat.csv",
  "sha256": "ae4efdfb3d7c0920e4f905573321bc29a7468a92ba369709c5fc235f187e09c7",
  "live_refresh_error": "ConnectionError: HTTPSConnectionPool(host='celestrak.org', port=443): Max retries exceeded with url: /pub/satcat.csv (Caused by NameResolutionError(\"HTTPSConnection(host='celestrak.org', port=443): Failed to resolve 'celestrak.org' ([Errno -3] Temporary failure in name resolution)\"))",
  "raw_rows": 70006,
  "raw_columns": 17
}
Raw shape: (70006, 17)


,OBJECT_NAME,OBJECT_ID,NORAD_CAT_ID,OBJECT_TYPE,OPS_STATUS_CODE,OWNER,LAUNCH_DATE,LAUNCH_SITE,DECAY_DATE,PERIOD,INCLINATION,APOGEE,PERIGEE,RCS,DATA_STATUS_CODE,ORBIT_CENTER,ORBIT_TYPE
0,SL-1 R/B,1957-001A,1,R/B,D,CIS,1957-10-04,TYMSC,1957-12-01,96.19,65.10,938.0,214.0,20.420,NaN,EA,IMP
1,SPUTNIK 1,1957-001B,2,PAY,D,CIS,1957-10-04,TYMSC,1958-01-03,96.10,65.00,1080.0,64.0,NaN,NaN,EA,IMP
2,SPUTNIK 2,1957-002A,3,PAY,D,CIS,1957-11-03,TYMSC,1958-04-14,103.74,65.33,1659.0,211.0,0.080,NaN,EA,IMP
3,EXPLORER 1,1958-001A,4,PAY,D,US,1958-02-01,AFETR,1970-03-31,88.48,33.15,215.0,183.0,NaN,NaN,EA,IMP
4,VANGUARD 1,1958-002B,5,PAY,NaN,US,1958-03-17,AFETR,NaN,132.59,34.25,3818.0,653.0,0.122,NaN,EA,ORB


## 5. Validate, clean, and filter the real catalogue

The final analysis focuses on objects that are:

- Earth-centred (`ORBIT_CENTER == "EA"`),
- currently in orbit (`ORBIT_TYPE == "ORB"` and no decay date),
- equipped with complete period, inclination, apogee, and perigee values, and
- within the study's Low Earth Orbit altitude band of 160 to 2,000 km based on mean altitude.

This is a reproducible study filter, not a statement that every object in this band is physically close to every other object.


In [4]:
def normalize_object_type(value: object) -> str:
    text = str(value).strip().upper()
    if text in {'PAY', 'PAYLOAD'}:
        return 'PAY'
    if text in {'DEB', 'DEBRIS'}:
        return 'DEB'
    if text in {'R/B', 'RB', 'ROCKET BODY'}:
        return 'R/B'
    return 'UNK'


df = df_raw.copy()

numeric_columns = ['NORAD_CAT_ID', 'PERIOD', 'INCLINATION', 'APOGEE', 'PERIGEE', 'RCS']
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

for column in ['LAUNCH_DATE', 'DECAY_DATE']:
    df[column] = pd.to_datetime(df[column], errors='coerce')

for column in ['OBJECT_NAME', 'OBJECT_ID', 'OWNER', 'ORBIT_CENTER', 'ORBIT_TYPE']:
    df[column] = df[column].fillna('UNKNOWN').astype(str).str.strip()

df['OBJECT_TYPE'] = df['OBJECT_TYPE'].apply(normalize_object_type)
df = df.drop_duplicates(subset='NORAD_CAT_ID', keep='last').copy()

# Engineered orbital features used in the models.
df['MEAN_ALTITUDE'] = (df['APOGEE'] + df['PERIGEE']) / 2.0
df['ORBIT_SPREAD'] = df['APOGEE'] - df['PERIGEE']
df['ALTITUDE_RATIO'] = df['PERIGEE'] / df['APOGEE'].replace(0, np.nan)

required_numeric = ['PERIOD', 'INCLINATION', 'APOGEE', 'PERIGEE'] + FEATURE_COLUMNS
valid_mask = (
    df['ORBIT_CENTER'].eq('EA')
    & df['ORBIT_TYPE'].eq('ORB')
    & df['DECAY_DATE'].isna()
    & df[required_numeric].notna().all(axis=1)
    & df['PERIGEE'].gt(0)
    & df['APOGEE'].ge(df['PERIGEE'])
    & df['MEAN_ALTITUDE'].between(LEO_MIN_ALTITUDE_KM, LEO_MAX_ALTITUDE_KM)
)

analysis_df = df.loc[valid_mask].copy().reset_index(drop=True)

if len(analysis_df) < 100:
    raise ValueError('Too few real records remained after filtering. Review the data source and filters.')

cleaning_summary = pd.DataFrame({
    'stage': [
        'Raw SATCAT rows',
        'Unique NORAD catalogue records',
        'Final current Earth-orbiting LEO analysis records',
    ],
    'rows': [len(df_raw), len(df), len(analysis_df)],
})

print(cleaning_summary.to_string(index=False))
print('\nFinal object types:')
print(analysis_df['OBJECT_TYPE'].value_counts().to_string())
analysis_df.head()


                                            stage  rows
                                  Raw SATCAT rows 70006
                   Unique NORAD catalogue records 70006
Final current Earth-orbiting LEO analysis records 28489

Final object types:
OBJECT_TYPE
PAY    17295
DEB    10134
R/B     1011
UNK       49


,OBJECT_NAME,OBJECT_ID,NORAD_CAT_ID,OBJECT_TYPE,OPS_STATUS_CODE,OWNER,LAUNCH_DATE,LAUNCH_SITE,DECAY_DATE,PERIOD,INCLINATION,APOGEE,PERIGEE,RCS,DATA_STATUS_CODE,ORBIT_CENTER,ORBIT_TYPE,MEAN_ALTITUDE,ORBIT_SPREAD,ALTITUDE_RATIO
0,VANGUARD 2,1959-001A,11,PAY,NaN,US,1959-02-17,AFETR,NaT,120.95,32.88,2895.0,552.0,0.3931,NaN,EA,ORB,1723.5,2343.0,0.190674
1,VANGUARD R/B,1959-001B,12,R/B,NaN,US,1959-02-17,AFETR,NaT,125.37,32.91,3286.0,553.0,0.5266,NaN,EA,ORB,1919.5,2733.0,0.168290
2,VANGUARD 3,1959-007A,20,PAY,NaN,US,1959-09-18,AFETR,NaT,123.93,33.35,3205.0,507.0,0.6412,NaN,EA,ORB,1856.0,2698.0,0.158190
3,EXPLORER 7,1959-009A,22,PAY,NaN,US,1959-10-13,AFETR,NaT,94.15,50.24,530.0,424.0,0.5003,NaN,EA,ORB,477.0,106.0,0.800000
4,TIROS 1,1960-002B,29,PAY,-,US,1960-04-01,AFETR,NaT,97.38,48.38,648.0,618.0,0.8030,NaN,EA,ORB,633.0,30.0,0.953704


## 6. Dataset overview and paper-ready descriptive figures


In [5]:
object_type_counts = (
    analysis_df['OBJECT_TYPE']
    .value_counts()
    .rename_axis('OBJECT_TYPE')
    .reset_index(name='COUNT')
)
owner_counts = (
    analysis_df['OWNER']
    .value_counts()
    .rename_axis('OWNER')
    .reset_index(name='COUNT')
)

print('Rows used in final analysis:', len(analysis_df))
print('Unique owners:', analysis_df['OWNER'].nunique())
print('Median mean altitude (km):', round(analysis_df['MEAN_ALTITUDE'].median(), 2))
object_type_counts


Rows used in final analysis: 28489
Unique owners: 84
Median mean altitude (km): 606.5


,OBJECT_TYPE,COUNT
0,PAY,17295
1,DEB,10134
2,R/B,1011
3,UNK,49


In [6]:
fig, ax = plt.subplots()
ax.bar(object_type_counts['OBJECT_TYPE'], object_type_counts['COUNT'])
ax.set_title('Current LEO catalogue records by object type')
ax.set_xlabel('Object type')
ax.set_ylabel('Number of records')
fig.tight_layout()
fig.savefig(OUTPUT_FIGURE_DIR / 'object_type_distribution.png', dpi=180, bbox_inches='tight')
plt.show()


/tmp/ipykernel_1916/382863428.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
plot_sample = analysis_df.sample(min(10_000, len(analysis_df)), random_state=RANDOM_STATE)

fig, ax = plt.subplots()
for object_type, group in plot_sample.groupby('OBJECT_TYPE'):
    ax.scatter(
        group['INCLINATION'],
        group['MEAN_ALTITUDE'],
        s=10,
        alpha=0.45,
        label=object_type,
    )
ax.set_title('Real CelesTrak LEO orbital regimes')
ax.set_xlabel('Inclination (degrees)')
ax.set_ylabel('Mean altitude (km)')
ax.legend(title='Object type')
fig.tight_layout()
fig.savefig(OUTPUT_FIGURE_DIR / 'altitude_vs_inclination_by_type.png', dpi=180, bbox_inches='tight')
plt.show()


/tmp/ipykernel_1916/216014020.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Pilot-versus-full real-catalogue comparison

The previous project used a very small demonstration dataset. To preserve the timeline's scaling comparison without returning to synthetic data, this notebook creates a reproducible 30-record **pilot sample drawn from the real cleaned SATCAT catalogue** and compares it with the full real catalogue. The pilot results are for workflow comparison only. The final paper should report the full-catalogue results.


In [8]:
def stratified_pilot_sample(frame: pd.DataFrame, n: int) -> pd.DataFrame:
    fractions = frame['OBJECT_TYPE'].value_counts(normalize=True)
    parts = []
    remaining = n
    types = list(fractions.index)
    for index, object_type in enumerate(types):
        available = frame[frame['OBJECT_TYPE'].eq(object_type)]
        if index == len(types) - 1:
            take = min(remaining, len(available))
        else:
            take = max(1, int(round(n * fractions[object_type])))
            take = min(take, len(available), remaining)
        if take > 0:
            parts.append(available.sample(take, random_state=RANDOM_STATE + index))
            remaining -= take
    pilot = pd.concat(parts, ignore_index=True)
    if len(pilot) < n:
        extra_pool = frame.loc[~frame['NORAD_CAT_ID'].isin(pilot['NORAD_CAT_ID'])]
        extra = extra_pool.sample(min(n - len(pilot), len(extra_pool)), random_state=RANDOM_STATE)
        pilot = pd.concat([pilot, extra], ignore_index=True)
    return pilot.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)


def evaluate_kmeans(frame: pd.DataFrame, label: str, k_values: list[int]) -> pd.DataFrame:
    scaler = StandardScaler()
    matrix = scaler.fit_transform(frame[FEATURE_COLUMNS])
    rows = []
    for k in k_values:
        if k >= len(frame):
            continue
        model = KMeans(
            n_clusters=k,
            random_state=RANDOM_STATE,
            n_init=20,
            max_iter=500,
        )
        labels = model.fit_predict(matrix)
        sample_size = min(SILHOUETTE_SAMPLE_SIZE, len(frame))
        score = silhouette_score(
            matrix,
            labels,
            sample_size=sample_size if sample_size < len(frame) else None,
            random_state=RANDOM_STATE,
        )
        rows.append({
            'DATASET': label,
            'K': k,
            'SILHOUETTE_SCORE': score,
            'INERTIA': model.inertia_,
            'N_RECORDS': len(frame),
        })
    return pd.DataFrame(rows)


pilot_df = stratified_pilot_sample(analysis_df, PILOT_SAMPLE_SIZE)
pilot_evaluation = evaluate_kmeans(pilot_df, 'Real 30-record pilot sample', list(range(2, 7)))
full_evaluation = evaluate_kmeans(analysis_df, 'Full real LEO catalogue', K_VALUES)
comparison_evaluation = pd.concat([pilot_evaluation, full_evaluation], ignore_index=True)

comparison_evaluation


,DATASET,K,SILHOUETTE_SCORE,INERTIA,N_RECORDS
0,Real 30-record pilot sample,2,0.451142,79.519832,30
1,Real 30-record pilot sample,3,0.419198,57.691960,30
2,Real 30-record pilot sample,4,0.451726,37.574888,30
3,Real 30-record pilot sample,5,0.462823,26.014669,30
4,Real 30-record pilot sample,6,0.432813,20.501234,30
5,Full real LEO catalogue,2,0.432200,86811.756840,28489
6,Full real LEO catalogue,3,0.451455,56787.473536,28489
7,Full real LEO catalogue,4,0.499078,38928.915708,28489
8,Full real LEO catalogue,5,0.517885,30032.701688,28489
9,Full real LEO catalogue,6,0.513340,24266.074094,28489


In [9]:
fig, ax = plt.subplots()
for dataset_name, group in comparison_evaluation.groupby('DATASET'):
    ax.plot(group['K'], group['SILHOUETTE_SCORE'], marker='o', label=dataset_name)
ax.set_title('Pilot versus full-catalogue cluster separation')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Silhouette score')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_FIGURE_DIR / 'pilot_vs_full_silhouette.png', dpi=180, bbox_inches='tight')
plt.show()


/tmp/ipykernel_1916/1997524347.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Standardize the final orbital features

Scaling is required because period is measured in minutes, inclination in degrees, and altitude-related variables in kilometres or ratios. StandardScaler gives each selected feature approximately zero mean and unit standard deviation before distance-based modelling.


In [10]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(analysis_df[FEATURE_COLUMNS])

scaled_feature_summary = pd.DataFrame({
    'FEATURE': FEATURE_COLUMNS,
    'SCALED_MEAN': X_scaled.mean(axis=0),
    'SCALED_STD': X_scaled.std(axis=0),
})
scaled_feature_summary


,FEATURE,SCALED_MEAN,SCALED_STD
0,PERIOD,6.384883e-17,1.0
1,INCLINATION,3.830930e-16,1.0
2,MEAN_ALTITUDE,1.596221e-16,1.0
3,ORBIT_SPREAD,3.192442e-17,1.0
4,ALTITUDE_RATIO,8.300348e-16,1.0


## 9. K-Means clustering and silhouette-based model selection


In [11]:
cluster_evaluation = full_evaluation.copy().sort_values('K').reset_index(drop=True)
BEST_K = int(cluster_evaluation.loc[cluster_evaluation['SILHOUETTE_SCORE'].idxmax(), 'K'])
BEST_SILHOUETTE = float(cluster_evaluation['SILHOUETTE_SCORE'].max())

print(f'Selected k = {BEST_K}')
print(f'Best sampled silhouette score = {BEST_SILHOUETTE:.4f}')
cluster_evaluation


Selected k = 5
Best sampled silhouette score = 0.5179


,DATASET,K,SILHOUETTE_SCORE,INERTIA,N_RECORDS
0,Full real LEO catalogue,2,0.432200,86811.756840,28489
1,Full real LEO catalogue,3,0.451455,56787.473536,28489
2,Full real LEO catalogue,4,0.499078,38928.915708,28489
3,Full real LEO catalogue,5,0.517885,30032.701688,28489
4,Full real LEO catalogue,6,0.513340,24266.074094,28489
5,Full real LEO catalogue,7,0.514369,21079.624958,28489
6,Full real LEO catalogue,8,0.513914,18965.702216,28489


In [12]:
fig, ax = plt.subplots()
ax.plot(cluster_evaluation['K'], cluster_evaluation['SILHOUETTE_SCORE'], marker='o')
ax.axvline(BEST_K, linestyle='--', label=f'Selected k={BEST_K}')
ax.set_title('K-Means silhouette-score evaluation')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Silhouette score')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_FIGURE_DIR / 'kmeans_silhouette_scores.png', dpi=180, bbox_inches='tight')
plt.show()


/tmp/ipykernel_1916/4143545389.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
fig, ax = plt.subplots()
ax.plot(cluster_evaluation['K'], cluster_evaluation['INERTIA'], marker='o')
ax.set_title('K-Means inertia (elbow diagnostic)')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Inertia')
fig.tight_layout()
fig.savefig(OUTPUT_FIGURE_DIR / 'kmeans_inertia.png', dpi=180, bbox_inches='tight')
plt.show()


/tmp/ipykernel_1916/4005693936.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
kmeans = KMeans(
    n_clusters=BEST_K,
    random_state=RANDOM_STATE,
    n_init=20,
    max_iter=500,
)
analysis_df['CLUSTER'] = kmeans.fit_predict(X_scaled)

cluster_summary = (
    analysis_df.groupby('CLUSTER')
    .agg(
        N_OBJECTS=('NORAD_CAT_ID', 'size'),
        MEAN_PERIOD=('PERIOD', 'mean'),
        MEAN_INCLINATION=('INCLINATION', 'mean'),
        MEAN_ALTITUDE=('MEAN_ALTITUDE', 'mean'),
        MEAN_ORBIT_SPREAD=('ORBIT_SPREAD', 'mean'),
        PAYLOADS=('OBJECT_TYPE', lambda values: int((values == 'PAY').sum())),
        DEBRIS=('OBJECT_TYPE', lambda values: int((values == 'DEB').sum())),
        ROCKET_BODIES=('OBJECT_TYPE', lambda values: int((values == 'R/B').sum())),
    )
    .reset_index()
)
cluster_summary.round(3)


,CLUSTER,N_OBJECTS,MEAN_PERIOD,MEAN_INCLINATION,MEAN_ALTITUDE,MEAN_ORBIT_SPREAD,PAYLOADS,DEBRIS,ROCKET_BODIES
0,0,3126,103.225,88.535,910.178,298.459,138,2845,133
1,1,10271,98.461,92.249,684.037,39.893,4559,5121,561
2,2,10925,94.341,50.577,486.388,4.137,10706,130,82
3,3,3480,112.098,84.454,1321.877,77.886,1803,1505,170
4,4,687,113.069,65.984,1364.579,1311.424,89,533,65


## 10. PCA visualization

PCA reduces the five standardized features to two displayed components. The explained-variance percentage describes how much variation is retained by the 2D visualization. It is not a model-accuracy score and does not validate collision prediction.


In [15]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
analysis_df['PCA1'] = X_pca[:, 0]
analysis_df['PCA2'] = X_pca[:, 1]

PCA_VARIANCE_PC1 = float(pca.explained_variance_ratio_[0])
PCA_VARIANCE_PC2 = float(pca.explained_variance_ratio_[1])
PCA_VARIANCE_TOTAL = float(pca.explained_variance_ratio_.sum())

print(f'PC1 explained variance: {PCA_VARIANCE_PC1:.2%}')
print(f'PC2 explained variance: {PCA_VARIANCE_PC2:.2%}')
print(f'Total variance retained in 2D: {PCA_VARIANCE_TOTAL:.2%}')


PC1 explained variance: 60.82%
PC2 explained variance: 23.01%
Total variance retained in 2D: 83.83%


In [16]:
pca_sample = analysis_df.sample(min(12_000, len(analysis_df)), random_state=RANDOM_STATE)

fig, ax = plt.subplots()
for cluster_id, group in pca_sample.groupby('CLUSTER'):
    ax.scatter(group['PCA1'], group['PCA2'], s=9, alpha=0.45, label=f'Cluster {cluster_id}')
ax.set_title('PCA visualization of real orbital-feature clusters')
ax.set_xlabel('Principal component 1')
ax.set_ylabel('Principal component 2')
ax.legend(title='K-Means cluster', ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT_FIGURE_DIR / 'pca_cluster_map.png', dpi=180, bbox_inches='tight')
plt.show()


/tmp/ipykernel_1916/3129890559.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Nearest-neighbour analysis and congestion scoring

For each object, the model finds the five nearest *other* objects in standardized feature space. The initial nearest-neighbour output includes the object itself at distance zero, so that self-match is removed before interpretation.

The earlier `(28, 4)` shape was not a scientific result. It simply meant 28 rows and four requested neighbours. In this notebook, the neighbour-distance matrix has one row per real analysed object and five columns after the self-match is removed.

The final congestion score is a **0-to-100 percentile rank** based on average nearest-neighbour distance. Smaller average distance receives a higher score. This avoids infinite values when multiple catalogue records have identical feature vectors.


In [17]:
neighbor_model = NearestNeighbors(n_neighbors=REQUESTED_NEIGHBORS + 1, algorithm='auto')
neighbor_model.fit(X_scaled)
all_distances, all_indices = neighbor_model.kneighbors(X_scaled)

# Remove the first column, which is each object's distance to itself.
neighbor_distances = all_distances[:, 1:]
neighbor_indices = all_indices[:, 1:]

analysis_df['MIN_NN_DISTANCE'] = neighbor_distances.min(axis=1)
analysis_df['AVG_NN_DISTANCE'] = neighbor_distances.mean(axis=1)
analysis_df['CONGESTION_SCORE'] = (
    analysis_df['AVG_NN_DISTANCE']
    .rank(method='max', pct=True, ascending=False)
    .mul(100.0)
)

score_percentile = analysis_df['CONGESTION_SCORE']
analysis_df['CONGESTION_CATEGORY'] = pd.cut(
    score_percentile,
    bins=[-np.inf, 50, 80, 95, np.inf],
    labels=['Low', 'Moderate', 'High', 'Very High'],
    include_lowest=True,
).astype(str)

print('Distance matrix including self:', all_distances.shape)
print('Distance matrix after removing self:', neighbor_distances.shape)
print('Congestion-score range:', round(analysis_df['CONGESTION_SCORE'].min(), 2), 'to', round(analysis_df['CONGESTION_SCORE'].max(), 2))


Distance matrix including self: (28489, 6)
Distance matrix after removing self: (28489, 5)
Congestion-score range: 0.0 to 100.0


In [18]:
top_congested = analysis_df.nlargest(25, 'CONGESTION_SCORE')[
    [
        'OBJECT_NAME', 'NORAD_CAT_ID', 'OBJECT_TYPE', 'OWNER', 'CLUSTER',
        'MEAN_ALTITUDE', 'INCLINATION', 'AVG_NN_DISTANCE',
        'CONGESTION_SCORE', 'CONGESTION_CATEGORY',
    ]
].copy()

top_congested


,OBJECT_NAME,NORAD_CAT_ID,OBJECT_TYPE,OWNER,CLUSTER,MEAN_ALTITUDE,INCLINATION,AVG_NN_DISTANCE,CONGESTION_SCORE,CONGESTION_CATEGORY
6911,TERRASAR-X,31698,PAY,GER,1,508.5,97.45,0.0,100.0,Very High
8383,TANDEM-X,36605,PAY,GER,1,508.5,97.45,0.0,100.0,Very High
8562,GLOBALSTAR M079,37188,PAY,GLOB,3,1413.5,52.01,0.0,100.0,Very High
8563,GLOBALSTAR M074,37189,PAY,GLOB,3,1413.5,52.00,0.0,100.0,Very High
8564,GLOBALSTAR M076,37190,PAY,GLOB,3,1413.5,52.01,0.0,100.0,Very High
8566,GLOBALSTAR M075,37192,PAY,GLOB,3,1413.5,52.00,0.0,100.0,Very High
8567,GLOBALSTAR M073,37193,PAY,GLOB,3,1413.5,52.01,0.0,100.0,Very High
8805,GLOBALSTAR M091,37741,PAY,GLOB,3,1413.5,52.00,0.0,100.0,Very High
8806,GLOBALSTAR M085,37742,PAY,GLOB,3,1413.5,52.01,0.0,100.0,Very High
8807,GLOBALSTAR M081,37743,PAY,GLOB,3,1413.5,52.00,0.0,100.0,Very High


In [19]:
plot_top = top_congested.sort_values('CONGESTION_SCORE')
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(plot_top['OBJECT_NAME'], plot_top['CONGESTION_SCORE'])
ax.set_title('Highest relative feature-space congestion scores')
ax.set_xlabel('Congestion percentile score (0-100)')
ax.set_ylabel('Object')
fig.tight_layout()
fig.savefig(OUTPUT_FIGURE_DIR / 'top_congestion_scores.png', dpi=180, bbox_inches='tight')
plt.show()


/tmp/ipykernel_1916/1985182435.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Long-form nearest-neighbour table


In [20]:
source_positions = np.repeat(np.arange(len(analysis_df)), REQUESTED_NEIGHBORS)
neighbor_positions = neighbor_indices.reshape(-1)
neighbor_ranks = np.tile(np.arange(1, REQUESTED_NEIGHBORS + 1), len(analysis_df))
flat_distances = neighbor_distances.reshape(-1)

source_names = analysis_df["OBJECT_NAME"].to_numpy()
source_ids = analysis_df["NORAD_CAT_ID"].astype(int).to_numpy()
source_types = analysis_df["OBJECT_TYPE"].to_numpy()
source_owners = analysis_df["OWNER"].to_numpy()

nearest_neighbor_table = pd.DataFrame({
    "OBJECT_NAME": source_names[source_positions],
    "NORAD_CAT_ID": source_ids[source_positions],
    "OBJECT_TYPE": source_types[source_positions],
    "OWNER": source_owners[source_positions],
    "NEIGHBOR_RANK": neighbor_ranks,
    "NEIGHBOR_NAME": source_names[neighbor_positions],
    "NEIGHBOR_NORAD_CAT_ID": source_ids[neighbor_positions],
    "NEIGHBOR_OBJECT_TYPE": source_types[neighbor_positions],
    "NEIGHBOR_OWNER": source_owners[neighbor_positions],
    "FEATURE_DISTANCE": flat_distances,
})

print("Nearest-neighbour table shape:", nearest_neighbor_table.shape)
nearest_neighbor_table.head(10)


Nearest-neighbour table shape: (142445, 10)


,OBJECT_NAME,NORAD_CAT_ID,OBJECT_TYPE,OWNER,NEIGHBOR_RANK,NEIGHBOR_NAME,NEIGHBOR_NORAD_CAT_ID,NEIGHBOR_OBJECT_TYPE,NEIGHBOR_OWNER,FEATURE_DISTANCE
0,VANGUARD 2,11,PAY,US,1,VANGUARD DEB,1576,DEB,US,0.483892
1,VANGUARD 2,11,PAY,US,2,VANGUARD DEB,69999,DEB,US,0.753564
2,VANGUARD 2,11,PAY,US,3,FREGAT DEB,45965,DEB,CIS,0.884062
3,VANGUARD 2,11,PAY,US,4,FREGAT DEB,46107,DEB,CIS,0.925643
4,VANGUARD 2,11,PAY,US,5,FREGAT DEB,45820,DEB,CIS,0.967124
5,VANGUARD R/B,12,R/B,US,1,VANGUARD 3,20,PAY,US,0.336459
6,VANGUARD R/B,12,R/B,US,2,FREGAT DEB,46003,DEB,CIS,0.825866
7,VANGUARD R/B,12,R/B,US,3,FREGAT DEB,45831,DEB,CIS,0.869628
8,VANGUARD R/B,12,R/B,US,4,FREGAT DEB,45815,DEB,CIS,0.935131
9,VANGUARD R/B,12,R/B,US,5,FREGAT DEB,52959,DEB,CIS,0.983070


## 13. Debris-to-payload feature-space proximity screening

For each debris record, the model identifies the nearest payload in the same standardized feature space. This is a monitoring-priority table, not evidence of a physical conjunction. The objects may not be near one another at the same time.


In [21]:
payload_mask = analysis_df['OBJECT_TYPE'].eq('PAY')
debris_mask = analysis_df['OBJECT_TYPE'].eq('DEB')

payload_df = analysis_df.loc[payload_mask].copy()
debris_df = analysis_df.loc[debris_mask].copy()
X_payload = X_scaled[payload_mask.to_numpy()]
X_debris = X_scaled[debris_mask.to_numpy()]

if payload_df.empty or debris_df.empty:
    debris_payload_pairs = pd.DataFrame()
else:
    payload_neighbor_model = NearestNeighbors(n_neighbors=1)
    payload_neighbor_model.fit(X_payload)
    pair_distances, pair_indices = payload_neighbor_model.kneighbors(X_debris)

    nearest_payload_positions = pair_indices[:, 0]
    pair_distance_values = pair_distances[:, 0]
    nearest_payloads = payload_df.iloc[nearest_payload_positions].reset_index(drop=True)
    debris_reset = debris_df.reset_index(drop=True)

    debris_payload_pairs = pd.DataFrame({
        'DEBRIS_NAME': debris_reset['OBJECT_NAME'],
        'DEBRIS_NORAD_CAT_ID': debris_reset['NORAD_CAT_ID'].astype(int),
        'DEBRIS_OWNER': debris_reset['OWNER'],
        'PAYLOAD_NAME': nearest_payloads['OBJECT_NAME'],
        'PAYLOAD_NORAD_CAT_ID': nearest_payloads['NORAD_CAT_ID'].astype(int),
        'PAYLOAD_OWNER': nearest_payloads['OWNER'],
        'FEATURE_DISTANCE': pair_distance_values,
    })
    debris_payload_pairs['PAIR_PRIORITY_PERCENTILE'] = (
        debris_payload_pairs['FEATURE_DISTANCE']
        .rank(method='average', pct=True, ascending=False)
        .mul(100.0)
    )
    debris_payload_pairs = debris_payload_pairs.sort_values('FEATURE_DISTANCE').reset_index(drop=True)

print('Debris-payload pair rows:', len(debris_payload_pairs))
debris_payload_pairs.head(20)


Debris-payload pair rows: 10134


,DEBRIS_NAME,DEBRIS_NORAD_CAT_ID,DEBRIS_OWNER,PAYLOAD_NAME,PAYLOAD_NORAD_CAT_ID,PAYLOAD_OWNER,FEATURE_DISTANCE,PAIR_PRIORITY_PERCENTILE
0,CZ-2F DEB,69673,PRC,PRC TEST SPACECRAFT 4,67689,PRC,0.000000,99.970397
1,CZ-6A DEB,65547,PRC,YAOGAN-40 03C,65546,PRC,0.000000,99.970397
2,COSMOS 1275 DEB,39959,CIS,COSMOS 1535,14679,CIS,0.000000,99.970397
3,YZ-1S DEB,43644,PRC,KINEIS-5D,62082,FR,0.000000,99.970397
4,FREGAT DEB,40078,CIS,MKA-PN 2 (RELEC),40070,CIS,0.000000,99.970397
5,COSMOS 2251 DEB,34852,CIS,COSMOS 968,10520,CIS,0.000000,99.970397
6,DMSP 5D-2 F13 DEB,40396,US,DMSP 5D-2 F8 (USA 26),18123,US,0.000000,99.970397
7,OPS 7613 (P/L 5) DEB,43191,US,TIMATION 2,4257,US,0.000452,99.930926
8,COSMOS 2251 DEB,33760,CIS,COSMOS 1992,19769,CIS,0.000452,99.921058
9,COSMOS 1660 DEB,43313,CIS,COSMOS 1732,16593,CIS,0.000452,99.911190


## 14. Owner-level descriptive summary for the dashboard


In [22]:
owner_summary = (
    analysis_df.groupby('OWNER')
    .agg(
        N_OBJECTS=('NORAD_CAT_ID', 'size'),
        N_PAYLOADS=('OBJECT_TYPE', lambda values: int((values == 'PAY').sum())),
        N_DEBRIS=('OBJECT_TYPE', lambda values: int((values == 'DEB').sum())),
        MEDIAN_CONGESTION_SCORE=('CONGESTION_SCORE', 'median'),
        MEAN_CONGESTION_SCORE=('CONGESTION_SCORE', 'mean'),
        MAX_CONGESTION_SCORE=('CONGESTION_SCORE', 'max'),
    )
    .reset_index()
    .sort_values(['N_OBJECTS', 'MEAN_CONGESTION_SCORE'], ascending=[False, False])
)

# Owner summaries describe catalogue composition and relative crowding. They do not assign legal responsibility.
owner_summary.head(20)


,OWNER,N_OBJECTS,N_PAYLOADS,N_DEBRIS,MEDIAN_CONGESTION_SCORE,MEAN_CONGESTION_SCORE,MAX_CONGESTION_SCORE
81,US,15798,12602,2952,100.000000,74.654131,100.000000
59,PRC,5485,1323,3966,31.907052,33.318630,100.000000
14,CIS,4844,1219,3100,25.723964,27.817761,62.009899
78,UK,704,703,0,100.000000,91.790163,100.000000
74,TBD,231,213,0,55.154621,48.919268,61.939696
26,FR,168,105,42,38.492050,33.449502,60.447892
40,JPN,160,128,12,37.493419,31.615207,59.345712
32,IND,119,59,35,28.442557,29.706733,60.911229
38,IT,85,85,0,52.750184,49.194054,100.000000
27,GER,74,74,0,41.454597,41.514411,100.000000


## 15. Exploratory Random Forest classification

The main research workflow is unsupervised. Random Forest is included as a separate exploratory supervised model to test whether the selected orbital features contain information that distinguishes payloads from debris.

Only PAY and DEB records are used. The split is stratified, the training model uses balanced class weights, and performance is reported using accuracy, balanced accuracy, class-specific precision/recall/F1, macro F1, and a confusion matrix. Random Forest does not require feature scaling, so it uses the engineered features in their original units.


In [23]:
classification_df = analysis_df[analysis_df['OBJECT_TYPE'].isin(['PAY', 'DEB'])].copy()
X_classification = classification_df[FEATURE_COLUMNS]
y_classification = classification_df['OBJECT_TYPE']

X_train, X_test, y_train, y_test = train_test_split(
    X_classification,
    y_classification,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_classification,
)

random_forest = RandomForestClassifier(
    n_estimators=150,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    min_samples_leaf=2,
    max_depth=18,
    n_jobs=1,
)
random_forest.fit(X_train, y_train)
y_pred = random_forest.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred)
rf_balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
rf_report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
rf_report = pd.DataFrame(rf_report_dict).T.reset_index().rename(columns={'index': 'CLASS_OR_AVERAGE'})
rf_confusion = confusion_matrix(y_test, y_pred, labels=['PAY', 'DEB'])
rf_confusion_df = pd.DataFrame(
    rf_confusion,
    index=['Actual PAY', 'Actual DEB'],
    columns=['Predicted PAY', 'Predicted DEB'],
)
rf_feature_importance = pd.DataFrame({
    'FEATURE': FEATURE_COLUMNS,
    'IMPORTANCE': random_forest.feature_importances_,
}).sort_values('IMPORTANCE', ascending=False)

rf_metrics = pd.DataFrame({
    'METRIC': [
        'Accuracy',
        'Balanced accuracy',
        'Macro precision',
        'Macro recall',
        'Macro F1',
        'PAY precision',
        'PAY recall',
        'DEB precision',
        'DEB recall',
    ],
    'VALUE': [
        rf_accuracy,
        rf_balanced_accuracy,
        rf_report_dict['macro avg']['precision'],
        rf_report_dict['macro avg']['recall'],
        rf_report_dict['macro avg']['f1-score'],
        rf_report_dict['PAY']['precision'],
        rf_report_dict['PAY']['recall'],
        rf_report_dict['DEB']['precision'],
        rf_report_dict['DEB']['recall'],
    ],
})

print(rf_metrics.to_string(index=False, formatters={'VALUE': '{:.4f}'.format}))
print('\nConfusion matrix:')
print(rf_confusion_df)
rf_feature_importance


           METRIC  VALUE
         Accuracy 0.9571
Balanced accuracy 0.9567
  Macro precision 0.9520
     Macro recall 0.9567
         Macro F1 0.9542
    PAY precision 0.9732
       PAY recall 0.9584
    DEB precision 0.9308
       DEB recall 0.9550

Confusion matrix:
            Predicted PAY  Predicted DEB
Actual PAY           4144            180
Actual DEB            114           2420


,FEATURE,IMPORTANCE
3,ORBIT_SPREAD,0.339538
4,ALTITUDE_RATIO,0.280297
2,MEAN_ALTITUDE,0.148364
0,PERIOD,0.120858
1,INCLINATION,0.110943


In [24]:
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(rf_confusion, cmap='Blues')
ax.set_xticks([0, 1], labels=['Predicted PAY', 'Predicted DEB'])
ax.set_yticks([0, 1], labels=['Actual PAY', 'Actual DEB'])
for row in range(rf_confusion.shape[0]):
    for column in range(rf_confusion.shape[1]):
        ax.text(column, row, rf_confusion[row, column], ha='center', va='center')
ax.set_title('Random Forest confusion matrix')
fig.colorbar(image, ax=ax)
fig.tight_layout()
fig.savefig(OUTPUT_FIGURE_DIR / 'random_forest_confusion_matrix.png', dpi=180, bbox_inches='tight')
plt.show()


/tmp/ipykernel_1916/3993298383.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
importance_plot = rf_feature_importance.sort_values('IMPORTANCE')
fig, ax = plt.subplots()
ax.barh(importance_plot['FEATURE'], importance_plot['IMPORTANCE'])
ax.set_title('Random Forest feature importance')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
fig.tight_layout()
fig.savefig(OUTPUT_FIGURE_DIR / 'random_forest_feature_importance.png', dpi=180, bbox_inches='tight')
plt.show()


/tmp/ipykernel_1916/405471206.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 16. Export Streamlit-compatible tables and model summaries

The Streamlit app reads the files created below. The app does not contain hard-coded research results.


In [26]:
export_df = analysis_df.copy()
for column in ['LAUNCH_DATE', 'DECAY_DATE']:
    export_df[column] = export_df[column].dt.strftime('%Y-%m-%d')

DASHBOARD_COLUMNS = [
    'OBJECT_NAME', 'OBJECT_ID', 'NORAD_CAT_ID', 'OBJECT_TYPE', 'OPS_STATUS_CODE',
    'OWNER', 'LAUNCH_DATE', 'LAUNCH_SITE', 'DECAY_DATE', 'PERIOD', 'INCLINATION',
    'APOGEE', 'PERIGEE', 'RCS', 'DATA_STATUS_CODE', 'ORBIT_CENTER', 'ORBIT_TYPE',
    'MEAN_ALTITUDE', 'ORBIT_SPREAD', 'ALTITUDE_RATIO', 'CLUSTER', 'PCA1', 'PCA2',
    'MIN_NN_DISTANCE', 'AVG_NN_DISTANCE', 'CONGESTION_SCORE', 'CONGESTION_CATEGORY',
]

dashboard_ready_table = export_df[DASHBOARD_COLUMNS].copy()

model_summary = {
    'source_url': SATCAT_URL,
    'source_mode': provenance['source_mode'],
    'raw_rows': int(len(df_raw)),
    'analysis_rows': int(len(analysis_df)),
    'leo_altitude_range_km': [LEO_MIN_ALTITUDE_KM, LEO_MAX_ALTITUDE_KM],
    'features': FEATURE_COLUMNS,
    'selected_k': BEST_K,
    'best_silhouette_score': BEST_SILHOUETTE,
    'pca_explained_variance_pc1': PCA_VARIANCE_PC1,
    'pca_explained_variance_pc2': PCA_VARIANCE_PC2,
    'pca_explained_variance_total': PCA_VARIANCE_TOTAL,
    'nearest_neighbors_per_object': REQUESTED_NEIGHBORS,
    'random_forest_accuracy': float(rf_accuracy),
    'random_forest_balanced_accuracy': float(rf_balanced_accuracy),
    'random_forest_macro_f1': float(rf_report_dict['macro avg']['f1-score']),
    'random_forest_pay_precision': float(rf_report_dict['PAY']['precision']),
    'random_forest_pay_recall': float(rf_report_dict['PAY']['recall']),
    'random_forest_deb_precision': float(rf_report_dict['DEB']['precision']),
    'random_forest_deb_recall': float(rf_report_dict['DEB']['recall']),
    'scientific_scope': 'Feature-space congestion screening; not collision probability.',
}


timeline_coverage = pd.DataFrame([
    {"TIMELINE_REQUIREMENT": "Real catalogue ingestion and validation", "NOTEBOOK_SECTION": "Sections 4-5", "OUTPUT": "data_provenance.json; cleaning_summary.csv"},
    {"TIMELINE_REQUIREMENT": "Larger-catalogue clustering and comparison", "NOTEBOOK_SECTION": "Sections 7-10", "OUTPUT": "pilot_vs_full_cluster_evaluation.csv; cluster_evaluation.csv; cluster_summary.csv"},
    {"TIMELINE_REQUIREMENT": "Dashboard-ready data exports", "NOTEBOOK_SECTION": "Sections 11-16", "OUTPUT": "dashboard_ready_table.csv; owner_summary.csv; debris_payload_pairs.csv"},
    {"TIMELINE_REQUIREMENT": "Unsupervised models and metrics", "NOTEBOOK_SECTION": "Sections 8-13", "OUTPUT": "K-Means, silhouette score, PCA, nearest-neighbour tables"},
    {"TIMELINE_REQUIREMENT": "Exploratory supervised classification", "NOTEBOOK_SECTION": "Section 15", "OUTPUT": "random_forest_metrics.csv; confusion matrix; feature importance"},
    {"TIMELINE_REQUIREMENT": "Paper-ready figures and verified results", "NOTEBOOK_SECTION": "Sections 6, 9-10, 15, 18", "OUTPUT": "outputs/figures; model_summary.json"},
    {"TIMELINE_REQUIREMENT": "Streamlit dashboard compatibility", "NOTEBOOK_SECTION": "Sections 16-17", "OUTPUT": "validated dashboard_ready_table.csv and supporting tables"},
])

outputs = {
    'dashboard_ready_table.csv': dashboard_ready_table,
    'cleaning_summary.csv': cleaning_summary,
    'object_type_counts.csv': object_type_counts,
    'owner_summary.csv': owner_summary,
    'pilot_vs_full_cluster_evaluation.csv': comparison_evaluation,
    'cluster_evaluation.csv': cluster_evaluation,
    'cluster_summary.csv': cluster_summary,
    'nearest_neighbor_table.csv': nearest_neighbor_table,
    'debris_payload_pairs.csv': debris_payload_pairs,
    'random_forest_metrics.csv': rf_metrics,
    'random_forest_classification_report.csv': rf_report,
    'random_forest_confusion_matrix.csv': rf_confusion_df.reset_index().rename(columns={'index': 'ACTUAL_CLASS'}),
    'random_forest_feature_importance.csv': rf_feature_importance,
    'timeline_coverage.csv': timeline_coverage,
}

for filename, table in outputs.items():
    table.to_csv(OUTPUT_TABLE_DIR / filename, index=False)

with (OUTPUT_TABLE_DIR / 'model_summary.json').open('w', encoding='utf-8') as handle:
    json.dump(model_summary, handle, indent=2)

with (OUTPUT_TABLE_DIR / 'data_provenance.json').open('w', encoding='utf-8') as handle:
    json.dump(provenance, handle, indent=2)

print('Saved table outputs:')
for path in sorted(OUTPUT_TABLE_DIR.glob('*')):
    print('-', path.relative_to(PROJECT_ROOT))


Saved table outputs:
- outputs/tables/cleaning_summary.csv
- outputs/tables/cluster_evaluation.csv
- outputs/tables/cluster_summary.csv
- outputs/tables/dashboard_ready_table.csv
- outputs/tables/data_provenance.json
- outputs/tables/debris_payload_pairs.csv
- outputs/tables/model_summary.json
- outputs/tables/nearest_neighbor_table.csv
- outputs/tables/object_type_counts.csv
- outputs/tables/owner_summary.csv
- outputs/tables/pilot_vs_full_cluster_evaluation.csv
- outputs/tables/random_forest_classification_report.csv
- outputs/tables/random_forest_confusion_matrix.csv
- outputs/tables/random_forest_feature_importance.csv
- outputs/tables/random_forest_metrics.csv
- outputs/tables/timeline_coverage.csv


## 17. Validate the Streamlit data contract


In [27]:
required_dashboard_columns = {
    'OBJECT_NAME', 'NORAD_CAT_ID', 'OBJECT_TYPE', 'OWNER', 'CLUSTER',
    'PCA1', 'PCA2', 'MEAN_ALTITUDE', 'INCLINATION',
    'CONGESTION_SCORE', 'CONGESTION_CATEGORY',
}
missing_dashboard_columns = required_dashboard_columns.difference(dashboard_ready_table.columns)

if missing_dashboard_columns:
    raise AssertionError(f'Dashboard table is missing columns: {sorted(missing_dashboard_columns)}')
if dashboard_ready_table.empty:
    raise AssertionError('Dashboard table is empty.')
if not dashboard_ready_table['CONGESTION_SCORE'].between(0, 100).all():
    raise AssertionError('Congestion scores must remain between 0 and 100.')

print('Streamlit data contract passed.')
print('Dashboard table shape:', dashboard_ready_table.shape)
print('Dashboard file:', OUTPUT_TABLE_DIR / 'dashboard_ready_table.csv')


Streamlit data contract passed.
Dashboard table shape: (28489, 27)
Dashboard file: /mnt/data/coco_orbital_congestion_real_data/outputs/tables/dashboard_ready_table.csv


## 18. Paper-ready results summary

Run this cell after every final clean run and copy the verified values into the paper. Do not retain the old 28-record demonstration metrics.


In [28]:
print(f'Real raw SATCAT rows loaded: {len(df_raw):,}')
print(f'Current Earth-orbiting LEO records analysed: {len(analysis_df):,}')
print(f'Final K-Means clusters: k={BEST_K}')
print(f'Best silhouette score: {BEST_SILHOUETTE:.3f}')
print(f'PCA variance retained in two components: {PCA_VARIANCE_TOTAL:.2%}')
print(f'Random Forest accuracy: {rf_accuracy:.2%}')
print(f'Random Forest balanced accuracy: {rf_balanced_accuracy:.2%}')
print(f'PAY precision: {rf_report_dict["PAY"]["precision"]:.2%}')
print(f'PAY recall: {rf_report_dict["PAY"]["recall"]:.2%}')
print(f'DEB precision: {rf_report_dict["DEB"]["precision"]:.2%}')
print(f'DEB recall: {rf_report_dict["DEB"]["recall"]:.2%}')
print(f'Debris-payload screening pairs: {len(debris_payload_pairs):,}')


Real raw SATCAT rows loaded: 70,006
Current Earth-orbiting LEO records analysed: 28,489
Final K-Means clusters: k=5
Best silhouette score: 0.518
PCA variance retained in two components: 83.83%
Random Forest accuracy: 95.71%
Random Forest balanced accuracy: 95.67%
PAY precision: 97.32%
PAY recall: 95.84%
DEB precision: 93.08%
DEB recall: 95.50%
Debris-payload screening pairs: 10,134


## 19. Scientific interpretation and limitations

- The clusters represent broad patterns in selected standardized orbital features. They are not official orbital classes.
- The silhouette score evaluates cluster cohesion and separation. It is not classification accuracy.
- PCA is used for visualization. Explained variance is not evidence of collision-prediction reliability.
- The congestion score is a relative percentile within this filtered catalogue and feature definition.
- Nearest neighbours are close in feature space, not necessarily close in three-dimensional position at the same time.
- The owner field is useful for descriptive filtering and transparency, but it does not assign responsibility or legal liability.
- The Random Forest analysis is exploratory and should be described separately from the unsupervised workflow.
- Real conjunction analysis would require time-dependent state vectors, orbital propagation, covariance and uncertainty information, relative velocity, object size, and environmental effects such as atmospheric drag.

## 20. Timeline completion checklist

After this notebook is committed and run successfully, the remaining project work is:

1. use the exported tables in the Streamlit app,
2. update the paper with the full-catalogue metrics,
3. add final figure numbers and captions,
4. complete the Results and Discussion sections,
5. deploy the Streamlit app,
6. polish the GitHub README and reproducibility instructions, and
7. complete final JHSS formatting and submission checks.
